In [ ]:


!pip install jedi
!pip install huggingface-hub
!pip install -q llama-index
!pip install llama-index-core
!pip install -q llama-index-embeddings-huggingface
!pip install -q nest_asyncio
!pip install -q llama-index-retrievers-bm25
!pip install -q sentence-transformers
!pip install -U google-genai llama-index-llms-google-genai

In [ ]:
import os
!pip install -q pymupdf # Ensure pymupdf is installed before import
import pymupdf  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from llama_index.llms.google_genai import GoogleGenAI
import nest_asyncio

nest_asyncio.apply()

.
GOOGLE_API_KEY = "AIzaAQ.Ab8RN6L-VYPWiWlpetRj9hzehqzXisimdO_tNBwBADZ77fVrJw"  # Replace with your actual API key from Google AI Studio


os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY


if GOOGLE_API_KEY == "AIzaAQ.Ab8RN6L-VYPWiWlpetRj9hzehqzXisimdO_tNBwBADZ77fVrJw":
    print("⚠️ WARNING: The provided GOOGLE_API_KEY does not appear to be a valid Google API key format.")
    print("Please ensure you have replaced `\"YOUR_API_KEY_HERE\"` with your actual API key obtained from: https://aistudio.google.com/app/apikey")
    if len(GOOGLE_API_KEY) > 4:
      print(f"Current key starts with: {GOOGLE_API_KEY[:4]}...")
    else:
      print(f"Current key: {GOOGLE_API_KEY}")


!mkdir -p sample_docs

⚠️ WARNING: The provided GOOGLE_API_KEY does not appear to be a valid Google API key format.
Please ensure you have replaced `"YOUR_API_KEY_HERE"` with your actual API key obtained from: https://aistudio.google.com/app/apikey
Current key starts with: AIza...


In [ ]:
from google.colab import files
import os

def upload_pdf():
    """Upload a PDF file and return its path."""
    print("Please select a PDF file to upload:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        if filename.endswith('.pdf'):
            # Save to the sample_docs directory
            pdf_path = os.path.join("sample_docs", filename)

            # Create directory if it doesn't exist
            os.makedirs("sample_docs", exist_ok=True)

            # Save the file
            with open(pdf_path, 'wb') as f:
                f.write(uploaded[filename])

            print(f"PDF saved to {pdf_path}")
            return pdf_path
        else:
            print(f"File {filename} is not a PDF. Please upload a PDF file.")

    return None

In [ ]:
from google.colab import userdata
userdata.get('GOOGLE_API_KEY')

'AQ.Ab8RN6L-VYPWiWlpetRj9hzehqzXisimdO_tNBwBADZ77fVrJw'

In [ ]:
pdf_path = upload_pdf()

Please select a PDF file to upload:


Saving sample-sdf-document (Project 5).pdf to sample-sdf-document (Project 5).pdf
PDF saved to sample_docs/sample-sdf-document (Project 5).pdf


In [ ]:


def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file using PyMuPDF."""
    doc = pymupdf.open(pdf_path)

    # Extract text from all pages
    text = "\n".join([page.get_text() for page in doc])

    # Print some stats
    print(f"PDF: {pdf_path}")
    print(f"Number of pages: {len(doc)}")
    print(f"Extracted {len(text.split())} words from the PDF.")

    # Close the document
    doc.close()

    return text

In [ ]:

if pdf_path:
    text = extract_text_from_pdf(pdf_path)
    print(text[:500])  # Print first 500 characters

PDF: sample_docs/sample-sdf-document (Project 5).pdf
Number of pages: 3
Extracted 617 words from the PDF.
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could le


In [ ]:


from llama_index.core import Document
from typing import List

def load_pdf_with_pymupdf(pdf_path: str) -> List[Document]:
    """Load a PDF and convert it to LlamaIndex Document format using PyMuPDF."""
    # Open the PDF
    doc = pymupdf.open(pdf_path)

    # Extract text from each page
    documents = []

    for i, page in enumerate(doc):
        text = page.get_text()

        # Skip empty pages
        if not text.strip():
            continue

        # Create Document object with metadata
        documents.append(
            Document(
                text=text,
                metadata={
                    "file_name": os.path.basename(pdf_path),
                    "page_number": i + 1,
                    "total_pages": len(doc)
                }
            )
        )

    # Close the document
    doc.close()

    # Print stats
    print(f"Processed {pdf_path}:")
    print(f"Extracted {len(documents)} pages with content")

    return documents

In [ ]:

pdf_docs = load_pdf_with_pymupdf(pdf_path)

Processed sample_docs/sample-sdf-document (Project 5).pdf:
Extracted 3 pages with content


In [ ]:
!apt-get -qq install -y libarchive-dev && pip install -U libarchive
import libarchive
from llama_index.llms.google_genai import GoogleGenAI # Corrected import from Gemini
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Initialize Gemini LLM
llm = GoogleGenAI(model="gemini-1.5-flash-001") # Updated model name to a currently available version
Settings.llm = llm

# Initialize embedding model
embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-small-v2")
Settings.embed_model = embed_model # Corrected typo embed_mode to embed_model

def process_and_index_pdf(pdf_path):
    """Process a PDF and create both vector and keyword indices."""
    # Load documents
    documents = load_pdf_with_pymupdf(pdf_path)

    # Create vector index
    vector_index = VectorStoreIndex.from_documents(documents)

    print(f"Indexed {len(documents)} document chunks")

    return vector_index

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'Model is not found: models/gemini-1.5-flash-001 for api version v1beta', 'status': 'NOT_FOUND'}}

In [ ]:


index = process_and_index_pdf(pdf_path)

Processed sample_docs/sample-sdf-document (Project 5).pdf:
Extracted 3 pages with content
Indexed 3 document chunks


In [ ]:
# Creating a function that rewrites queries to improve search results.
# Simple query expansion function using Gemini
def expand_query(query: str, num_expansions: int = 3) -> list:
    """Expand a query to include related terms using Gemini."""
    prompt = f"""
    I need to search a pharmaceutical quality document with this query: "{query}"

    Please help me expand this query by generating {num_expansions} alternative versions that:
    1. Use different but related terminology
    2. Include relevant pharmaceutical/quality terms that might appear in a certificate or SDF
    3. Cover similar concepts but phrased differently

    Format your response as a list of alternative queries only, with no additional text.
    """

    response = llm.complete(prompt)

    # Extract the expanded queries
    expanded_queries = [line.strip() for line in response.text.split('\n') if line.strip()]

    # Add the original query if needed
    if query not in expanded_queries:
        expanded_queries = [query] + expanded_queries

    return expanded_queries

In [ ]:
#idk if we need thisssss
# WHAT WE'RE DOING:
# Testing query expansion with a sample pharmaceutical question.
#
# WHAT YOU'LL SEE:
# The original query plus expanded versions with related terms.

# Example usage:
expanded = expand_query("What test methods were used for quality control?")
for i, q in enumerate(expanded):
    print(f"{i+1}. {q}")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}

In [ ]:
# ============================================
# STEP 12: Build Query Expansion Engine
#
# WHAT WE'RE DOING:
# Building a more advanced query engine that automatically expands queries.
#
# WHY THIS MATTERS:
# QueryFusionRetriever generates multiple versions of your question and combines results.
#
# WHAT YOU'LL SEE:
# No output — the engine is ready to use.
# ============================================

from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever

# Function to create a query engine that uses query expansion
def create_query_expansion_engine(index):
    """Create a query engine that uses query expansion."""
    # First create multiple retrievers (base retriever)
    base_retriever = index.as_retriever(similarity_top_k=2)

    # Create a query fusion retriever
    fusion_retriever = QueryFusionRetriever(
        retrievers=[base_retriever],
        llm=llm,
        similarity_top_k=2,
        num_queries=3,  # Generate 3 queries per original query
        mode="reciprocal_rerank"  # Use reciprocal rank fusion
    )

    # Create the query engine with the fusion retriever
    query_engine = RetrieverQueryEngine.from_args(
        retriever=fusion_retriever,
        llm=llm,
        verbose=True
    )

    return query_engine

In [ ]:

expanded_query_engine = create_query_expansion_engine(index)
response = expanded_query_engine.query("What test methods were used for quality control?")
print(response)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/migrate-to-interactions).', 'status': 'NOT_FOUND'}}

In [ ]:

print("BM25 library was already installed in Step 0 — ready to use!")

BM25 library was already installed in Step 0 — ready to use!


In [ ]:


from llama_index.core import VectorStoreIndex
from llama_index.retrievers.bm25 import BM25Retriever

def create_hybrid_retriever(index, query, top_k=2):
    """Create a hybrid retrieval approach combining vector and keyword search."""
    # Method 1: Vector retrieval (semantic search)
    vector_retriever = index.as_retriever(similarity_top_k=top_k)
    vector_nodes = vector_retriever.retrieve(query)

    # Method 2: BM25 retrieval (keyword-based search)
    # Get all nodes from the index
    nodes = [node for node in index.docstore.docs.values()]
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=nodes,
        similarity_top_k=top_k
    )
    keyword_nodes = bm25_retriever.retrieve(query)

    # Combine results (simple approach)
    all_nodes = []
    all_nodes.extend(vector_nodes)
    all_nodes.extend(keyword_nodes)

    # Remove duplicates
    unique_nodes = []
    seen_ids = set()
    for node in all_nodes:
        if node.node_id not in seen_ids:
            unique_nodes.append(node)
            seen_ids.add(node.node_id)

    # Sort by score (higher is better)
    sorted_nodes = sorted(unique_nodes, key=lambda x: x.score if hasattr(x, 'score') else 0.0, reverse=True)

    # Limit to top results
    top_nodes = sorted_nodes[:top_k]

    return top_nodes

ModuleNotFoundError: No module named 'llama_index'

In [ ]:


query = "What test methods were used for quality control?"
top_k_values = [2, 5, 10]

In [ ]:


for top_k in top_k_values:
    print(f"\n{'='*60}")
    print(f"Results for top_k = {top_k}")
    print(f"{'='*60}")
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)
    for i, node in enumerate(nodes):
        print(f"\nResult {i+1} (Score: {node.score:.3f}):")
        print(node.get_text())
        print("-" * 50)


Results for top_k = 2

Result 1 (Score: 0.332):
Best regards,
Mike Toner
Product Manager
michaeltoner@cytiva.com

Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quality management system.
--------------------------------------------------

Result 2 (Score: 0.311):
5 bar, 5 min hold
Pass
Visual Inspection
No defects visible
Pass
Package Integrity
Sealed,
--------------------------------------------------

Results for top_k = 5

Result 1 (Score: 0.332):
Best regards,
Mike Toner
Product Manager
michaeltoner@cytiva.com

Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quality management system.
--------------------------------------------------

Result 2 (Score: 0.311):
5 bar, 5 min hold
Pass
Visual Inspection
No defects visible
Pass
Package Integrity
Sealed,
--------------------------------------------------

Result 3 (Score: 0.260):
derived ingredients or in compliance with EMA/410/01.  
Countr

In [ ]:

hybrid_nodes = create_hybrid_retriever(index, "What are the storage conditions for this product?")
for i, node in enumerate(hybrid_nodes):
    print(f"Result {i+1} (Score: {node.score:.4f}):")
    print(node.get_text())
    print("-" * 40)

DEBUG:bm25s:Building index from IDs objects


Result 1 (Score: 0.8224):
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could lead to 
brittleness or cracking of the plastic connectors. However, the operating temperature of ÄKTA ready flow kits is +2 C to 
+40 C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of
damage to the kit during setup and handling.
Description
Part Number
Operating Temperature
High Flow Kit F, Modified, ÄKTA ready
29477427
+2 C to +40 C
High Flow Gradient C, Modified, ÄKTA ready
29184612
+2 C to +40 C
Operating I

In [ ]:


from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.schema import NodeWithScore

# Create a reranker
def rerank_results(nodes, query, top_n=2):
    """Rerank retrieved nodes using the Sentence Transformer reranker."""
    # Create the reranker
    reranker = SentenceTransformerRerank(
        model="cross-encoder/ms-marco-MiniLM-L-6-v2",
        top_n=top_n
    )

    # Rerank the nodes
    reranked_nodes = reranker.postprocess_nodes(
        nodes,
        query_str=query
    )

    return reranked_nodes

# Function to demonstrate the reranking process
def demonstrate_reranking(index, query, top_k=3):
    """Demonstrate the reranking process on retrieval results."""
    # First retrieve more nodes than we need
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)

    print(f"Query: {query}")
    print("\nOriginal Retrieval Order:")
    for i, node in enumerate(nodes):
        print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")

    # Now rerank them
    reranked_nodes = rerank_results(nodes, query, top_n=2)

    print("\nAfter Reranking:")
    for i, node in enumerate(reranked_nodes):
        print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")

    # Creating comparison dataframe
    results = []

    # Original ranking
    for i, node in enumerate(nodes):
        results.append({
            "Stage": "Original Retrieval",
            "Rank": i + 1,
            "Score": node.score,
            "Content": node.get_text()[:150] + "...",
            "Page": node.metadata.get("page_number", "Unknown")
        })

    # Reranked
    for i, node in enumerate(reranked_nodes):
        results.append({
            "Stage": "After Reranking",
            "Rank": i + 1,
            "Score": node.score,
            "Content": node.get_text()[:150] + "...",
            "Page": node.metadata.get("page_number", "Unknown")
        })

    results_df = pd.DataFrame(results)
    display(results_df)

    return results_df

# Example usage:
reranking_demo = demonstrate_reranking(index, "What sterilization method was used?", top_k=3)

Query: What sterilization method was used?

Original Retrieval Order:
1. (Score: 0.7847) - Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 900...
2. (Score: 0.7729) - Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quali...
3. (Score: 0.7643) - Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄK...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


After Reranking:
1. (Score: -11.1414) - Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 900...
2. (Score: -11.2425) - Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quali...


,Stage,Rank,Score,Content,Page
0,Original Retrieval,1,-11.141409,Cytiva\ncytiva.com\nCertificate of Quality\nTh...,3
1,Original Retrieval,2,-11.242511,Certificate of Quality \nThis product is manuf...,2
2,Original Retrieval,3,-11.248030,"Cytiva\n100 Results Way\nMarlborough, MA 01752...",1
3,After Reranking,1,-11.141409,Cytiva\ncytiva.com\nCertificate of Quality\nTh...,3
4,After Reranking,2,-11.242511,Certificate of Quality \nThis product is manuf...,2


In [ ]:


from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, QueryBundle

def build_rag_pipeline(index):
    """Build a simple but effective RAG pipeline with hybrid retrieval and reranking."""

    # Get all nodes from the index's docstore
    nodes = list(index.docstore.docs.values())

    # Determine safe top_k value (number of nodes to retrieve)
    # Must be at least 1 and no more than the number of available nodes
    num_nodes = len(nodes)
    safe_top_k = min(2, max(1, num_nodes))

    print(f"Index contains {num_nodes} nodes, using top_k={safe_top_k}")

    # Create a hybrid retriever combining vector and keyword search
    # First, get the vector retriever (for semantic understanding)
    vector_retriever = index.as_retriever(
        similarity_top_k=safe_top_k  # Retrieve top 2 most similar chunks
    )

    # Next, create a BM25 retriever (for keyword matching)
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=nodes,
        similarity_top_k=safe_top_k  # Retrieve top 2 most similar chunks
    )

    # Create a proper hybrid retriever class
    class HybridRetriever(BaseRetriever):
        """Hybrid retriever that combines vector and keyword search results."""

        def __init__(self, vector_retriever, keyword_retriever, top_k=2):
            """Initialize with vector and keyword retrievers."""
            self.vector_retriever = vector_retriever
            self.keyword_retriever = keyword_retriever
            self.top_k = top_k
            super().__init__()

        def _retrieve(self, query_bundle, **kwargs):
            """Retrieve from both retrievers and combine results."""
            # Get results from both retrievers
            vector_nodes = self.vector_retriever.retrieve(query_bundle)
            keyword_nodes = self.keyword_retriever.retrieve(query_bundle)

            # Combine all nodes
            all_nodes = list(vector_nodes) + list(keyword_nodes)

            # Remove duplicates (by node_id)
            unique_nodes = {}
            for node in all_nodes:
                if node.node_id not in unique_nodes:
                    unique_nodes[node.node_id] = node

            # Sort by score (higher is better)
            sorted_nodes = sorted(
                unique_nodes.values(),
                key=lambda x: x.score if hasattr(x, 'score') else 0.0,
                reverse=True
            )

            return sorted_nodes[:self.top_k]  # Return top results

    # Create our hybrid retriever instance
    hybrid_retriever = HybridRetriever(
        vector_retriever=vector_retriever,
        keyword_retriever=bm25_retriever,
        top_k=safe_top_k
    )

    # Step 2: Create a reranker to prioritize the most relevant chunks
    if num_nodes > 1:
        reranker = SentenceTransformerRerank(
            model="cross-encoder/ms-marco-MiniLM-L-6-v2",
            top_n=min(2, num_nodes)  # Keep only top 2 results after reranking
        )
        node_postprocessors = [reranker]
    else:
        node_postprocessors = []


    # Step 3: Build the query engine
    query_engine = RetrieverQueryEngine.from_args(
        retriever=hybrid_retriever,
        llm=llm,
        node_postprocessors=node_postprocessors
    )

    return query_engine

In [ ]:

index = process_and_index_pdf(pdf_path)
rag_engine = build_rag_pipeline(index)
response = rag_engine.query("What test methods were used for quality control?")
response = rag_engine.query("What are the storage conditions specified in the certificate?")
print('\nFinal Response:\n ---------------------- \n')
print('\nFinal Response:\n ---------------------- \n')
print(response)

In [ ]:
!pip install --upgrade gradio

In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs="text"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5acbc0155f29a91e4d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def analyze(text):
    return len(text.split()), text.upper()

demo = gr.Interface(
    fn=analyze,
    inputs="textbox",
    outputs=["number", "textbox"]
)

In [ ]:
import gradio as gr
def calculate_powers_and_factorial(number):
    """Takes a number, calculates its square, cube, and factorial,

    and returns them as a labeled dictionary.
    """
    # Cast to integer for factorial calculation
    int_num = int(number)

    # Factorial is only defined for non-negative integers
    if int_num < 0:
        fact_result = "Undefined for negative numbers"
    else:
        try:
            fact_result = math.factorial(int_num)
        except OverflowError:
            fact_result = "Number too large"

    return {
        "Square (x²)": number**2,
        "Cube (x³)": number**3,
        "Factorial (x!)": fact_result,
    }

def greet(name):
    return "Hello " + name + "!"

demo = gr.Interface(
    fn=calculate_powers_and_factorial,
    inputs=gr.Number(label="Enter a Number", precision=0),
    outputs=gr.JSON(label="Results"),
    title="Number Calculator",
    description="Enter a whole number to calculate its square, cube, and factorial.",
)
demo.launch(share=True)  # 🚀 One line = public link


In [ ]:


def process_pdf(file):
    return "PDF processed successfully!"

def handle_chat(message, history):
    return history + [(message, "This is a mock answer.")]

with gr.Blocks(title="Step 5: Full Functioning UI") as demo:
    gr.Markdown("### Step 5: Connected UI")

    with gr.Column(scale=2):
        chatbot = gr.Chatbot(label="Chat History", height=300)
        user_input = gr.Textbox(
            placeholder="Ask a question about your document...",
            label="Your Question"
        )
        send_btn = gr.Button("📤 Send")
        clear_btn = gr.Button("🗑️ Clear Chat")

    with gr.Column(scale=1):
        pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
        process_btn = gr.Button("🔄 Process Document")

NameError: name 'gr' is not defined

In [ ]:
import gradio as gr

def process_pdf(file):
    return "PDF processed successfully!"

def handle_chat(message, history):
    return history + [(message, "This is a mock answer.")]

with gr.Blocks(title="Step 5: Full Functioning UI") as demo:
    gr.Markdown("### Step 5: Connected UI")

    with gr.Column(scale=2):
        chatbot = gr.Chatbot(label="Chat History", height=300)
        user_input = gr.Textbox(
            placeholder="Ask a question about your document...",
            label="Your Question"
        )
        send_btn = gr.Button("📤 Send")
        clear_btn = gr.Button("🗑️ Clear Chat")

    with gr.Column(scale=1):
        pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
        process_btn = gr.Button("🔄 Process Document")

    process_btn.click(process_pdf, inputs=pdf_input, outputs=None)
    send_btn.click(handle_chat, inputs=[user_input, chatbot], outputs=chatbot)
    user_input.submit(handle_chat, inputs=[user_input, chatbot], outputs=chatbot)
    clear_btn.click(lambda: [], outputs=chatbot)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b227070b8f0d01ce49.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
